# TP6 - Analyse des Résultats de Mapping et Quantification des Gènes
## vendredi 25/09/2025

Ce notebook couvre l'analyse complète des résultats de mapping RNA-seq, incluant l'évaluation de la qualité du mapping et la quantification au niveau des gènes.

<div class="alert alert-info">
<b>Exigences d'Environnement :</b><br>
- 5 CPUs par étudiant<br>
- 10 GB RAM par étudiant<br>
- Outils : samtools, QUALIMAP, featureCounts, MultiQC
</div>

## 1. Évaluation de la Qualité du Mapping

Cette première section se concentre sur l'évaluation de la qualité des fichiers BAM issus du mapping avec STAR, en utilisant samtools et QUALIMAP pour générer des statistiques détaillées.

## Fichiers et Données Requis

<div class="alert alert-warning">
<b>Fichiers d'Entrée Requis :</b>
1. Données RNA-seq paired-end (format fastq.gz)
   - Localisation : `/srv/data/meg-m1-a2b/blumenthal-2014/wholeData/fastq/raw/`
   - Format : `*_1.fastq.gz` et `*_2.fastq.gz` pour les reads paired-end

2. Fichiers du Génome de Référence :
   - Génome de souris (GRCm39)
   - Annotation GTF : `/srv/data/meg-m1-a2b/blumenthal-2014/wholeData/Genomes/Mmu/GRCm39/extracted/genome_annotation-M35.gtf`

3. Fichiers Générés Précédemment :
   - Fichiers BAM du mapping STAR
   - Résultats fastp dans `./Analysis_chr7/3-mapping/fastp/`
</div>

<div class="alert alert-info">
<b>Structure de Sortie :</b>
Les résultats seront organisés dans des répertoires spécifiques aux outils :
- `./Analysis_chr7/3-mapping/samtools/` - Statistiques d'alignement
- `./Analysis_chr7/3-mapping/qualimap/` - Métriques QC RNA-seq
- `./Analysis_chr7/3-mapping/multiqc/` - Rapports de qualité combinés
</div>

## Génome de Référence et Annotation

Cette section couvre le téléchargement et l'indexation des fichiers du génome de référence. Ces étapes sont généralement effectuées lors de la configuration de l'environnement.

<div class="alert alert-info">
<b>Fichiers de Référence :</b>
- Génome : Mus musculus GRCm39
- Source : Ensembl Release 109
- Annotation : genome_annotation-M35.gtf
</div>

<div class="alert alert-warning">
<b>Note :</b> Les commandes suivantes sont fournies à titre de référence et de documentation. Elles sont conservées au format brut car elles sont généralement exécutées lors de la configuration de l'environnement.
</div>

## Évaluation de la Qualité des Lectures Mappées

Nous utiliserons plusieurs outils pour évaluer la qualité de nos reads mappés :

1. **samtools** : Statistiques d'alignement de base
   - Comptage des reads
   - Qualité de mapping
   - Distribution de la taille d'insertion

2. **QUALIMAP** : Métriques RNA-seq détaillées
   - Couverture des gènes
   - Distribution des reads
   - Couverture des transcrits

3. **MultiQC** : Génération de rapports combinés
   - Agrège les résultats de tous les outils
   - Crée des visualisations interactives

<div class="alert alert-info">
<b>Liens de Documentation :</b>
- [Manuel Samtools](http://www.htslib.org/doc/samtools.html)
- [Documentation QUALIMAP](http://qualimap.conesalab.org/doc_html/index.html)
- [Documentation MultiQC](https://multiqc.info/)
</div>

### Configuration des Répertoires de Résultats

D'abord, nous créerons les répertoires nécessaires pour organiser nos résultats.

In [ ]:

## Code cell 1 ##
# Créer les répertoires de résultats
mkdir -p ./Analysis_chr7/3-mapping/samtools ./Analysis_chr7/3-mapping/qualimap ./Analysis_chr7/3-mapping/multiqc

### Traitement des Deux Premiers Échantillons

Nous démontrerons le pipeline d'analyse sur les deux premiers échantillons. Le même processus sera appliqué à tous les échantillons plus tard.

<div class="alert alert-info">
<b>Paramètres Importants :</b>
- samtools stats : Génère des statistiques d'alignement complètes
- QUALIMAP rnaseq :
  - java-mem-size=8G : Allocation mémoire (adaptée aux 10GB disponibles)
  - Fichier GTF : Requis pour l'analyse basée sur les gènes
</div>

In [ ]:

## Code cell 2 ##
# Obtenir les deux premiers échantillons
samples=$(ls ./Analysis_chr7/3-mapping/bam/*_1.bam | head -n 2)

for sample in $samples; do
    base=$(basename $sample _1.bam)
    echo "Traitement de l'échantillon : $base"

    echo "Exécution de samtools stats..."
    samtools stats ./Analysis_chr7/3-mapping/bam/${base}.bam > ./Analysis_chr7/3-mapping/samtools/${base}_stats.txt

    echo "Exécution de QUALIMAP..."
    qualimap rnaseq \
        -bam ./Analysis_chr7/3-mapping/bam/${base}.bam \
        -gtf /srv/data/meg-m1-a2b/blumenthal-2014/wholeData/Genomes/Mmu/GRCm39/extracted/genome_annotation-M35.gtf \
        --java-mem-size=8G \
        -outdir ./Analysis_chr7/3-mapping/qualimap/${base}
done

### Traitement de Tous les Échantillons

Les commandes suivantes (au format brut) montrent comment traiter tous les échantillons du jeu de données.

## Génération de Rapport MultiQC

MultiQC combine les rapports de qualité de tous les échantillons en un rapport complet.

<div class="alert alert-info">
<b>Fonctionnalités MultiQC :</b>
- Combine les rapports de plusieurs outils
- Crée des visualisations interactives
- Permet une comparaison facile des échantillons
- Génère un rapport HTML

<b>Sources d'Entrée :</b>
- Résultats samtools stats
- Rapports RNA-seq QUALIMAP
</div>

In [ ]:

## Code cell 3 ##
# Générer le rapport MultiQC
echo "Génération du rapport MultiQC..."
multiqc \
    ./Analysis_chr7/3-mapping/samtools/ \
    ./Analysis_chr7/3-mapping/qualimap/ \
    -o ./Analysis_chr7/3-mapping/multiqc/ \
    --title "Rapport de Qualité RNA-seq Souris" \
    --comment "Généré pour le cours PS4-2024"


echo "Analyse QUALIMAP pour les échantillons du jeu de données complet..."

mkdir -p .Analysis_wholeData/3-mapping/qualimap/

for sample in /srv/data/meg-m1-a2b/blumenthal-2014/wholeData/3-bam/*.bam; do
    base=$(basename $sample .bam)
    echo "Traitement de l'échantillon complet : $base"

    qualimap rnaseq \
        -bam $sample \
        -gtf /srv/data/meg-m1-a2b/blumenthal-2014/wholeData/Genomes/Mmu/GRCm39/extracted/genome_annotation-M35.gtf \
        --java-mem-size=8G \
        -outdir .Analysis_wholeData/3-mapping/qualimap/${base}
done



Dans cette section, nous construirons une boucle pour exécuter Qualimap sur tous les échantillons étape par étape. Cette approche aide à comprendre comment automatiser les tâches répétitives en bash.

D'abord, regardons la commande Qualimap de base que nous avons utilisée pour un seul échantillon :

In [ ]:

## Code cell 4 ##
# Exemple pour un seul échantillon
qualimap bamqc     -bam ./Analysis_chr7/3-mapping/samtools/sample1_sorted.bam     -outdir ./Analysis_chr7/3-mapping/qualimap/sample1     --java-mem-size=8G


Dans la commande ci-dessus, nous pouvons identifier deux parties principales qui changent pour chaque échantillon :
1. Chemin du fichier BAM d'entrée (`./Analysis_chr7/3-mapping/samtools/sample1_sorted.bam`)
2. Chemin du répertoire de sortie (`./Analysis_chr7/3-mapping/qualimap/sample1`)

Le nom de l'échantillon est la variable clé qui change dans les deux chemins.


D'abord, nous créerons une liste de nos noms d'échantillons. Nous pouvons faire cela en listant les fichiers BAM triés et en extrayant les noms d'échantillons :

In [ ]:

## Code cell 5 ##
# Lister les noms d'échantillons
ls ./Analysis_chr7/3-mapping/samtools/*_sorted.bam | sed 's|./Analysis_chr7/3-mapping/samtools/||' | sed 's|_sorted.bam||'


Maintenant, créons une structure de boucle simple qui itérera sur nos échantillons :

In [ ]:

## Code cell 6 ##
# Structure de boucle de base
for sample in $(ls ./Analysis_chr7/3-mapping/samtools/*_sorted.bam | sed 's|./Analysis_chr7/3-mapping/samtools/||' | sed 's|_sorted.bam||')
do
    echo "Traitement de l'échantillon : $sample"
done


Maintenant nous ajouterons la commande Qualimap à l'intérieur de notre boucle, en utilisant des variables pour les parties spécifiques à l'échantillon :

In [ ]:

## Code cell 7 ##
# Boucle complète avec commande Qualimap
for sample in $(ls ./Analysis_chr7/3-mapping/samtools/*_sorted.bam | sed 's|./Analysis_chr7/3-mapping/samtools/||' | sed 's|_sorted.bam||')
do
    echo "Traitement de l'échantillon : $sample"

    mkdir -p ./Analysis_chr7/3-mapping/qualimap/${sample}

    qualimap bamqc         -bam ./Analysis_chr7/3-mapping/samtools/${sample}_sorted.bam         -outdir ./Analysis_chr7/3-mapping/qualimap/${sample}         --java-mem-size=8G
done


1. La commande `mkdir -p` assure que notre répertoire de sortie existe
2. Nous utilisons `${sample}` pour séparer clairement le nom de variable dans les chemins
3. Le paramètre `--java-mem-size=8G` est important en raison des contraintes mémoire (10GB par étudiant, allocation adaptée)
4. L'utilisation d'instructions `echo` aide à suivre le progrès pendant l'exécution

Essayez de modifier la boucle pour :
1. Ajouter une vérification d'erreur
2. Inclure un pourcentage de progression
3. Sauvegarder un fichier de log de l'analyse

## 2. Quantification au Niveau des Gènes

Cette section couvre la quantification des gènes à partir des fichiers BAM mappés, en utilisant featureCounts pour obtenir des comptages de reads par gène.

## Fichiers et Données Requis

<div class="alert alert-warning">
<b>Fichiers d'Entrée Requis :</b>
1. Fichiers BAM mappés (issus de STAR)
   - Localisation : `./Analysis_chr7/3-mapping/bam/`
   - Format : fichiers BAM triés par coordonnées

2. Fichier d'Annotation GTF :
   - Annotation des gènes de souris (GRCm39)
   - Localisation : `/srv/data/Genomes/Mmu/GRCm39/gencode.vM37.primary_assembly.annotation.gtf.gz`
</div>

<div class="alert alert-info">
<b>Structure de Sortie :</b>
Les résultats seront organisés dans :
- `./Analysis_chr7/3-mapping/counts/` - Comptages de gènes featureCounts
- `./Analysis_chr7/3-mapping/multiqc/` - Rapports de qualité combinés
</div>

## Quantification au Niveau des Gènes avec featureCounts

featureCounts est un outil efficace pour compter les reads qui se chevauchent avec les caractéristiques génomiques (gènes, exons, etc.).

<div class="alert alert-info">
<b>Paramètres Importants de featureCounts :</b>
- `-p` : Données paired-end
- `-s 0` : Données non-orientées (unstranded)
- `-T 4` : Utilise 4 threads
- `-a` : Fichier d'annotation GTF
- `-o` : Fichier de sortie

<b>Liens de Documentation :</b>
- [Manuel featureCounts](http://subread.sourceforge.net/featureCounts.html)
- [Documentation MultiQC](https://multiqc.info/)
</div>

### Configuration des Répertoires de Résultats

Créons d'abord les répertoires nécessaires pour organiser nos résultats.

In [ ]:

## Code cell 8 ##
# Créer les répertoires de résultats
mkdir -p ./Analysis_chr7/3-mapping/counts ./Analysis_chr7/3-mapping/multiqc

### Traitement d'un Échantillon Unique

Nous démontrerons d'abord le processus de quantification sur un seul échantillon pour comprendre les étapes individuelles.

In [ ]:

## Code cell 9 ##
# Traiter le premier échantillon BAM
sample_bam=$(ls ./Analysis_chr7/3-mapping/bam/*.bam | head -n 1)
sample_name=$(basename $sample_bam .bam)

echo "Traitement de l'échantillon : $sample_name"

featureCounts \
    -p \
    -s 0 \
    -T 4 \
    -a /srv/data/Genomes/Mmu/GRCm39/gencode.vM37.primary_assembly.annotation.gtf.gz \
    -o ./Analysis_chr7/3-mapping/counts/${sample_name}_counts.txt \
    $sample_bam

echo "Résumé de l'assignation :"
cat ./Analysis_chr7/3-mapping/counts/${sample_name}_counts.txt.summary

### Traitement de Tous les Échantillons

Maintenant, nous traiterons tous les échantillons en une seule commande featureCounts pour créer une matrice de comptage complète.

In [ ]:

## Code cell 10 ##
# Traiter tous les échantillons ensemble
echo "Traitement de tous les échantillons BAM..."

featureCounts \
    -p \
    -s 0 \
    -T 4 \
    -a /srv/data/Genomes/Mmu/GRCm39/gencode.vM37.primary_assembly.annotation.gtf.gz \
    -o ./Analysis_chr7/3-mapping/counts/all_samples_counts.txt \
    ./Analysis_chr7/3-mapping/bam/*.bam

cut -f 1,7- ./Analysis_chr7/3-mapping/counts/all_samples_counts.txt | grep -v '^#' > ./Analysis_chr7/3-mapping/counts/counts_matrix.txt

echo "Matrice de comptage créée : ./Analysis_chr7/3-mapping/counts/counts_matrix.txt"
echo "Nombre de gènes quantifiés :"
wc -l ./Analysis_chr7/3-mapping/counts/counts_matrix.txt

### Interprétation des Résultats

<div class="alert alert-info">
<b>Fichiers de Sortie de featureCounts :</b>
- `*_counts.txt` : Fichier principal avec les comptages par gène
- `*_counts.txt.summary` : Résumé des statistiques d'assignation
- `counts_matrix.txt` : Matrice simplifiée pour l'analyse downstream

<b>Statistiques d'Assignation Importantes :</b>
- **Assigned** : Reads assignés avec succès aux gènes
- **Unassigned_NoFeatures** : Reads dans des régions sans annotation
- **Unassigned_Ambiguity** : Lectures chevauchant plusieurs gènes
</div>

In [ ]:

## Code cell 11 ##
# Vérifier les statistiques d'assignation
echo "=== Statistiques d'Assignation featureCounts ==="
cat ./Analysis_chr7/3-mapping/counts/all_samples_counts.txt.summary

echo -e "\n=== Aperçu de la Matrice de Comptage ==="
echo "Premières lignes de la matrice :"
head -n 5 ./Analysis_chr7/3-mapping/counts/counts_matrix.txt

echo -e "\n=== Informations sur les Fichiers Créés ==="
ls -lh ./Analysis_chr7/3-mapping/counts/

## Génération de Rapport MultiQC

MultiQC compile les rapports de qualité de featureCounts en un rapport HTML interactif.

In [ ]:

## Code cell 12 ##
# Générer le rapport MultiQC
echo "Génération du rapport MultiQC..."

multiqc \
    ./Analysis_chr7/3-mapping/counts/ \
    -o ./Analysis_chr7/3-mapping/multiqc/ \
    --title "Rapport de Quantification RNA-seq Souris" \
    --comment "Quantification au niveau des gènes avec featureCounts - Cours PS5-2024"

echo "Rapport MultiQC généré dans : ./Analysis_chr7/3-mapping/multiqc/"
ls -la ./Analysis_chr7/3-mapping/multiqc/

## Construction Progressive de Boucle pour featureCounts

Dans cette section, nous construirons une boucle pour traiter les échantillons individuellement, étape par étape. Cette approche aide à comprendre l'automatisation des tâches répétitives.

Regardons d'abord la commande featureCounts de base pour un seul échantillon :

In [ ]:

## Code cell 13 ##
# Exemple pour un seul échantillon
featureCounts -p -s 0 -T 4 \
    -a /srv/data/Genomes/Mmu/GRCm39/gencode.vM37.primary_assembly.annotation.gtf.gz \
    -o ./Analysis_chr7/3-mapping/counts/sample1_counts.txt \
    ./Analysis_chr7/3-mapping/bam/sample1.bam

### Étape 2 : Identifier les Parties Variables
Dans la commande ci-dessus, deux parties changent pour chaque échantillon :
1. Fichier BAM d'entrée (`./Analysis_chr7/3-mapping/bam/sample1.bam`)
2. Fichier de sortie (`./Analysis_chr7/3-mapping/counts/sample1_counts.txt`)

Le nom de l'échantillon est la variable clé qui change dans les deux chemins.

### Étape 3 : Créer une Liste des Échantillons
Créons une liste de nos fichiers BAM :

In [ ]:

## Code cell 14 ##
# Lister les fichiers BAM
ls ./Analysis_chr7/3-mapping/bam/*.bam

### Étape 4 : Construire la Boucle - Structure de Base
Créons une structure de boucle simple qui itérera sur nos échantillons :

In [ ]:

## Code cell 15 ##
# Structure de boucle de base
for bam_file in ./Analysis_chr7/3-mapping/bam/*.bam; do
    sample_name=$(basename $bam_file .bam)
    echo "Traitement de l'échantillon : $sample_name"
done

### Étape 5 : Ajouter la Commande featureCounts
Ajoutons maintenant la commande featureCounts à l'intérieur de notre boucle :

In [ ]:

## Code cell 16 ##
# Boucle complète avec featureCounts
for bam_file in ./Analysis_chr7/3-mapping/bam/*.bam; do
    sample_name=$(basename $bam_file .bam)
    echo "Traitement de l'échantillon : $sample_name"
    
    mkdir -p ./Analysis_chr7/3-mapping/counts/
    
    featureCounts -p -s 0 -T 4 \
        -a /srv/data/Genomes/Mmu/GRCm39/gencode.vM37.primary_assembly.annotation.gtf.gz \
        -o ./Analysis_chr7/3-mapping/counts/${sample_name}_counts.txt \
        $bam_file
        
    echo "Terminé : $sample_name"
done

### Notes Importantes :
1. La commande `mkdir -p` assure que notre répertoire de sortie existe
2. Nous utilisons `${sample_name}` pour séparer clairement le nom de variable dans les chemins
3. Le paramètre `-T 4` utilise 4 threads (adapté aux 5 CPUs disponibles)
4. L'utilisation d'instructions `echo` aide à suivre le progrès pendant l'exécution

Essayez de modifier la boucle pour :
1. Ajouter une vérification d'erreur
2. Inclure un compteur de progression
3. Sauvegarder un fichier de log de l'analyse

Les fichiers de comptage générés peuvent être utilisés pour :
- Analyse d'expression différentielle avec DESeq2 ou edgeR
- Analyse de voies métaboliques
- Visualisation des données d'expression

## 3. Résumé et Prochaines Étapes

1. **Statistiques de Mapping** : Évaluation de la qualité avec samtools et QUALIMAP
2. **Comptages de Gènes** : Quantification avec featureCounts
3. **Rapports de Qualité** : Compilation avec MultiQC

- `./Analysis_chr7/3-mapping/samtools/` - Statistiques samtools
- `./Analysis_chr7/3-mapping/qualimap/` - Rapports QUALIMAP
- `./Analysis_chr7/3-mapping/counts/` - Comptages featureCounts
- `./Analysis_chr7/3-mapping/multiqc/` - Rapports MultiQC

Les fichiers de comptage générés peuvent être utilisés pour :
- Analyse d'expression différentielle avec DESeq2 ou edgeR
- Analyse de voies métaboliques
- Visualisation des données d'expression

<div class="alert alert-success">
<b>Félicitations !</b> Vous avez terminé l'analyse des résultats de mapping RNA-seq et obtenu des comptages de gènes prêts pour l'analyse downstream.
</div>